In [173]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import matplotlib as plt

RAW_DATA_DIR = Path("../data/raw")

TREES_FILE = RAW_DATA_DIR / "ppr_tree_inventory_2025.geojson"
PHENOMETRICS_FILE = RAW_DATA_DIR / "PA phenometrics .csv"
ANCILLARY_FILE = RAW_DATA_DIR / "ancillary_individual_plant_data.csv"


In [174]:
trees = gpd.read_file(TREES_FILE)
phenometrics = gpd.read_file(PHENOMETRICS_FILE)
individual_trees = gpd.read_file(ANCILLARY_FILE)

In [175]:
# Get list of tree names in philly
unique_trees_philly = trees.drop_duplicates(subset=["tree_name"])
unique_trees_philly.set_index(["tree_name"])
philly_tree_names = unique_trees_philly[["tree_name"]]
philly_tree_names
# drop nulls
philly_tree_names = philly_tree_names.dropna()
# normalize all spaces to only one space
philly_tree_names["tree_name"] = philly_tree_names["tree_name"].str.split().str.join(' ')
# Make first character of tree names capitalized
philly_tree_names["tree_name"] = philly_tree_names["tree_name"].str.capitalize()
print(len(philly_tree_names))

296


In [176]:
# Make DF of trees in philly with scientific name and common name
philly_tree_names["scientific_name"] = philly_tree_names["tree_name"].str.split(r"\s*[-–—]\s*", regex=True).str[0]
philly_tree_names["common_name"] = philly_tree_names["tree_name"].str.split(r"\s*[-–—]\s*", regex=True).str[1]

hybrid_philly_trees = philly_tree_names[philly_tree_names["scientific_name"].str.contains(" x ", na=False)]
print(len(hybrid_philly_trees))
hybrid_philly_trees

# Remove " x " from some scientific names
philly_tree_names["scientific_name"] = philly_tree_names["scientific_name"].str.replace(" x ", " ")


12


In [177]:
# check for and drop duplicate scientific names
unique_scientific_names = philly_tree_names.drop_duplicates(subset=["scientific_name"])
print(len(unique_scientific_names))



253


In [178]:
#check for and drop duplicate common names
unique_common_names = philly_tree_names.drop_duplicates(subset=["common_name"])


In [179]:
# Create new column in phenometrics data combining genus & species
# Keep only unique scientific names
phenometrics["scientific_name"] = phenometrics["Genus"] + ' ' + phenometrics["Species"]
unique_phenometrics = phenometrics.drop_duplicates(subset=["scientific_name"])

# change common name title

unique_phenometrics = unique_phenometrics.rename(columns={"Common_Name": "common_name"})



In [180]:
# Look for matches across philly data & phenometrics
exact_matches_scientific = unique_scientific_names.merge(unique_phenometrics, on="scientific_name", how="inner")
print(len(exact_matches_scientific))




83


# So I have 83 tree from the philly data that are represented in the PA NPN data
# NEXT STEP:
# FIND HOW MANY OF THE TREES IN PHILLY ARE AMONG THESE SPECIES

In [181]:
trees["tree_name"] = trees["tree_name"].str.capitalize()
clean_trees = (
    trees
    .dropna(subset=["tree_name", "geometry"])  # geometry being non-null implies coords present
    .drop(columns=["tree_dbh", "year", "loc_x", "loc_y"])
    .loc[lambda d: d["tree_name"].str.strip().ne("")]
    .loc[lambda d: ~d["tree_name"].str.contains("unknown", case=False, na=False)]
    .copy()
)

# Collapse whitespace (this is what fixes "Prunus  yedoensis")
clean_trees["tree_name"] = (
    clean_trees["tree_name"].str.replace(r"\s+", " ", regex=True).str.strip()
)

# Split tree_name ONCE on dash-with-whitespace. Handles -, –, — without eating "crus-galli"
parts = clean_trees["tree_name"].str.split(r"\s+[-–—]\s+", n=1, expand=True, regex=True)
clean_trees["scientific_name"] = parts[0].str.replace(" x ", " ", regex=False)
clean_trees["common_name"] = parts[1]

# Genus / Species in a single split of the already-cleaned scientific_name
gs = clean_trees["scientific_name"].str.split(n=2, expand=True)
clean_trees["Genus"] = gs[0]
clean_trees["Species"] = gs[1]


In [182]:
# find how many matches there are citywide

philly_wide_matches= clean_trees.merge(exact_matches_scientific, on="scientific_name", how="inner")
print(len(philly_wide_matches)/ len(clean_trees))

0.46832405771986535


# ONLY 69,685 (46%) trees are represented in NPN PA data

In [183]:
# These are the ones that failed to match based on scientific name
non_match_scientific_names = unique_scientific_names[~unique_scientific_names["scientific_name"].isin(exact_matches_scientific["scientific_name"])]

non_match_scientific_names


,tree_name,scientific_name,common_name
0,Ginkgo biloba - ginkgo,Ginkgo biloba,ginkgo
1,Acer palmatum - japanese maple,Acer palmatum,japanese maple
6,Unknown unknown - unknown,Unknown unknown,unknown
7,Koelreuteria paniculata - goldenrain tree,Koelreuteria paniculata,goldenrain tree
9,Amelanchier species - other serviceberry,Amelanchier species,other serviceberry
...,...,...,...
125156,Sorbus aucuparia - european mountain ash,Sorbus aucuparia,european mountain ash
127966,Sorbus americana - american mountain ash,Sorbus americana,american mountain ash
142125,Acer henryii – henrys maple,Acer henryii,henrys maple
149494,Ulmus alata - winged elm,Ulmus alata,winged elm


In [184]:
# Next step, find how many of each missing tree is in philly
philly_trees_non_npn = clean_trees[clean_trees["scientific_name"].isin(non_match_scientific_names["scientific_name"])]

philly_misses_count = philly_trees_non_npn["scientific_name"].value_counts().reset_index().set_index("scientific_name")

philly_misses_count.head(40)

,count
scientific_name,
Platanus acerifolia,16172
Prunus species,7458
Zelkova serrata,4514
Syringa reticulata,3772
Ginkgo biloba,3712
Malus species,2758
Tilia species,2681
Acer campestre,2519
Crataegus species,1834


In [185]:
clean_trees["scientific_name"].value_counts().head(40)

scientific_name
Platanus acerifolia              16172
Acer rubrum                      10313
Prunus species                    7458
Pyrus calleryana                  7173
Gleditsia triacanthos             4993
Quercus rubra                     4822
Acer platanoides                  4666
Zelkova serrata                   4514
Syringa reticulata                3772
Ginkgo biloba                     3712
Cercis canadensis                 2882
Malus species                     2758
Tilia species                     2681
Prunus serrulata                  2546
Acer saccharum                    2527
Acer campestre                    2519
Quercus palustris                 2406
Crataegus species                 1834
Sophora japonica                  1833
Tilia cordata                     1809
Cladrastis kentukea               1713
Amelanchier species               1704
Acer species                      1697
Ulmus species                     1694
Liquidambar styraciflua           1590
Quercus b

In [186]:
%store clean_trees

Stored 'clean_trees' (GeoDataFrame)


In [187]:
clean_trees.to_file("../data/processed/philly_trees.geojson")
clean_trees

,objectid,tree_name,geometry,scientific_name,common_name,Genus,Species
0,1,Ginkgo biloba - ginkgo,POINT (-75.2105 39.98383),Ginkgo biloba,ginkgo,Ginkgo,biloba
1,2,Acer palmatum - japanese maple,POINT (-75.21053 39.98374),Acer palmatum,japanese maple,Acer,palmatum
2,3,Acer palmatum - japanese maple,POINT (-75.21041 39.98376),Acer palmatum,japanese maple,Acer,palmatum
3,4,Acer palmatum - japanese maple,POINT (-75.2106 39.98395),Acer palmatum,japanese maple,Acer,palmatum
4,5,Acer pseudoplatanus - sycamore maple,POINT (-75.21028 39.98376),Acer pseudoplatanus,sycamore maple,Acer,pseudoplatanus
...,...,...,...,...,...,...,...
151721,151722,Platanus x acerifolia - london planetree,POINT (-75.20873 39.94644),Platanus acerifolia,london planetree,Platanus,acerifolia
151722,151723,Platanus x acerifolia - london planetree,POINT (-75.20876 39.9466),Platanus acerifolia,london planetree,Platanus,acerifolia
151723,151724,Gleditsia triacanthos - honeylocust,POINT (-75.16226 39.99548),Gleditsia triacanthos,honeylocust,Gleditsia,triacanthos
151724,151725,Gleditsia triacanthos - honeylocust,POINT (-75.16221 39.9955),Gleditsia triacanthos,honeylocust,Gleditsia,triacanthos


In [188]:
nameless_trees = clean_trees.loc[clean_trees["tree_name"] == ""]
nameless_trees

,objectid,tree_name,geometry,scientific_name,common_name,Genus,Species
